# Modelica to PyPowSyBl: Dynamic Simulation Toolkit (Model Compilation)

### Industrialized Workflow
1. **Create Network:** Programmatically build the static loadflow structure via PyPowSyBl.
2. **Initialize Loadflow:** Define load and generation profiles.
3. **Run Loadflow:** Execute the base AC Loadflow using OpenLF.
4. **Compile Dynamic Models:** Compile the dynamic models with the Dynawo model generator instead of using the models precompiled in the Dynawo database: the standard generator model, and a variant of our own with another voltage regulator.
5. **Dynamic Models Setup:** Map the static generators to the compiled models.
6. **Physical & Initialization Parameters:** Build the `.PAR` files using physical parameters and Loadflow initialization references.
7. **Execute Dynamic Simulations:** Simulate the same event with the precompiled model, the compiled model and the compiled variant, and compare them.

## Step 0: Environment Setup (Dynawo Install & Config)

Before running dynamic simulations, we need the Dynawo engine installed and configured so PyPowSyBl can find it.

In [ ]:
# --- 1. ENVIRONMENT SETUP: CHECK DYNAWO INSTALLATION ---
import os
import re
import datetime


def verify_dynawo_installation():
    """Finds and validates the Dynawo installation path from the config file."""
    config_directory = os.path.expanduser("~/.itools")
    config_filepath = os.path.join(config_directory, "config.yml")
    dynawo_path = None

    # Check if config.yml exists and extract the homeDir value
    if os.path.exists(config_filepath):
        with open(config_filepath, "r") as file:
            content = file.read()
            match = re.search(r'homeDir:\s*["\']?([^"\'\n]+)["\']?', content)
            if match:
                dynawo_path = match.group(1)

    # Verify if a path was extracted and if it actually exists
    if not dynawo_path or not os.path.exists(dynawo_path):
        raise FileNotFoundError(
            f"Error! No valid Dynawo installation found on the system. "
            f"Detected path: '{dynawo_path}'. Make sure Dynawo is installed and properly configured in {config_filepath}."
        )

    print(f"Configuration correct! Dynawo installation validated at: {dynawo_path}")

    # Save the verified path in an environment variable for subsequent steps
    os.environ["DYNAWO_SYS_PATH"] = dynawo_path
    return dynawo_path, config_directory


# Execute verification
dynawo_sys_path, global_config_dir = verify_dynawo_installation()

In [ ]:
# --- 2. ENVIRONMENT SETUP: CONFIGURE TEMPLATES ---
def configure_environment_template(dynawo_path, config_directory):
    """Overwrites the PyPowSyBl configuration with the local project template."""
    target_config_filepath = os.path.join(config_directory, "config.yml")
    template_config_filepath = "./getting_started_data/config.yml"
    current_working_directory = os.getcwd()

    if not dynawo_path:
        raise ValueError("The Dynawo path is not defined. Please run the previous cell first.")

    # Create a backup of the existing configuration if it exists
    if os.path.exists(target_config_filepath):
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_filepath = os.path.join(config_directory, f"config_{timestamp}.yml")
        os.rename(target_config_filepath, backup_filepath)
        print(f"Backup created at: {backup_filepath}")

    with open(template_config_filepath, "r") as file:
        template_content = file.read()

    # Inject dynamic paths into the template
    template_content = template_content.replace('"WORKING_DIR/dynawo"', f'"{dynawo_path}"')
    template_content = template_content.replace("WORKING_DIR", current_working_directory)

    os.makedirs(config_directory, exist_ok=True)
    with open(target_config_filepath, "w") as file:
        file.write(template_content)

    print(f"\nDone! Template successfully configured and saved to {target_config_filepath}:\n")
    print(template_content)


# Execute configuration
configure_environment_template(dynawo_sys_path, global_config_dir)

In [ ]:
# --- 3. IMPORT LIBRARIES AND BASE CONFIGURATION ---
import contextlib
import json
import shutil
import subprocess
import time
import xml.etree.ElementTree as ET
from pathlib import Path

import pandas as pd
import pypowsybl as pp
import pypowsybl.dynamic as dyn
import matplotlib.pyplot as plt
from IPython.display import display

# Compilation toolchain of the toolkit: checks of the Dynawo distribution and of the compiler
from python_pypowsybl.Code.core.model_compiler import DynawoToolchain

# Ensure plots are displayed inline within the Jupyter environment
%matplotlib inline

print(f"PyPowSyBl version: {pp.__version__}")

## Step 1 & 2: Create Network and Initialize Loadflow Profiles

We construct the 2-bus network structure programmatically using the provided utility function.

In [ ]:
# --- 4. CREATE NETWORK AND INITIALIZE PROFILES ---
def build_tutorial_network() -> pp.network.Network:
    """
    Programmatically constructs a basic 2-bus static power network.
    """
    # Initialize an empty network using the correct PyPowSyBl function
    power_network = pp.network.load("getting_started_data/Base_Case.xiidm")

    # Create substations and voltage levels
    power_network.create_substations(
        id=["SUB_GEN", "SUB_LOAD"], name=["Generator Substation", "Load Substation"]
    )
    power_network.create_voltage_levels(
        id=["VL_GEN", "VL_LOAD"],
        name=["VL_GEN", "VL_LOAD"],
        substation_id=["SUB_GEN", "SUB_LOAD"],
        topology_kind=["BUS_BREAKER", "BUS_BREAKER"],
        nominal_v=[400.0, 400.0],
    )

    # Create buses
    power_network.create_buses(
        id=["BUS_GEN", "BUS_LOAD"],
        name=["BUS_GEN", "BUS_LOAD"],
        voltage_level_id=["VL_GEN", "VL_LOAD"],
    )

    # Add Generator (acting as Slack bus regulating voltage)
    power_network.create_generators(
        id="GEN_1",
        name="GEN_1",
        voltage_level_id="VL_GEN",
        bus_id="BUS_GEN",
        target_p=150.0,
        target_v=400.0,
        min_p=0.0,
        max_p=300.0,
        voltage_regulator_on=True,
    )

    # Add Load
    power_network.create_loads(
        id="LOAD_1",
        name="LOAD_1",
        voltage_level_id="VL_LOAD",
        bus_id="BUS_LOAD",
        p0=150.0,
        q0=50.0,
    )

    # Connect the buses with a transmission line
    power_network.create_lines(
        id="LINE_1",
        name="LINE_1",
        voltage_level1_id="VL_GEN",
        bus1_id="BUS_GEN",
        voltage_level2_id="VL_LOAD",
        bus2_id="BUS_LOAD",
        r=1.5,
        x=15.0,
        b1=0.0005,
        b2=0.0005,
    )

    return power_network


print("Creating power network structure...")
power_network = build_tutorial_network()

## Step 3: Run Loadflow

Execute the base Loadflow using PyPowSyBl's OpenLF. The Loadflow provider automatically identifies the Slack bus based on network topology.

In [ ]:
# --- 5. RUN STEADY-STATE LOADFLOW ---
def execute_and_display_loadflow(network: pp.network.Network):
    """Executes AC Load Flow analysis and displays the resulting bus metrics."""
    print("Running Loadflow (OpenLF)...")
    loadflow_parameters = pp.loadflow.Parameters()
    loadflow_results = pp.loadflow.run_ac(network, parameters=loadflow_parameters)

    print(f"Loadflow Status: {loadflow_results[0].status.name}")

    buses_dataframe = network.get_buses()
    display(buses_dataframe[["name", "v_mag", "v_angle"]])


# Execute loadflow
execute_and_display_loadflow(power_network)

## Step 4: Compile the Dynamic Models

Instead of using the dynamic models precompiled in the Dynawo model database (`ddb`), this notebook compiles them from their Modelica definition with the Dynawo model generator (`dynawo.sh jobs --generate-preassembled`), as described in the [compilation guide](../../../Documentation/compilation_guide.md). Two models are compiled:

- `GeneratorSynchronousThreeWindingsGoverNordicVRNordic`: the standard model of the Getting Started notebooks, compiled from its official definition. It replaces the precompiled model in a model database of this notebook, and must give the same results.
- `GeneratorSynchronousThreeWindingsGoverNordicVRProportional`: a variant of our own, with the same machine and governor, but the simplified proportional voltage regulator of the Dynawo library (`VRProportional`) instead of `VRNordic`.

Each model is described by an architecture file, compiled into a library (`.so`), and its description (`.desc.xml`, the parameters and variables of the model) is dumped from the library. Compiling requires a full Dynawo distribution (headers and on-the-fly model compilation), built with a compiler compatible with the C++ compiler of the system (see the guide). Each model takes about half a minute to compile; the compiled libraries are reused by the next runs.

In [ ]:
# --- 6. COMPILATION SETUP: TOOLCHAIN AND DIRECTORIES ---
# The toolchain checks that the Dynawo distribution can compile models with the C++
# compiler of the system, and prepares the environment of the Dynawo model generator
# (compiler with the C++ ABI of the Dynawo libraries, Boost of the distribution)
toolchain = DynawoToolchain.detect(dynawo_sys_path)
print(f"Dynawo version: {toolchain.dynawo_version()}")
print(f"Toolchain: {toolchain.description}")

# Every file of the compilation goes to the export directory of the notebook
COMPILATION_DIRECTORY = Path("grid_export_GettingStarted_I/compiled_models").resolve()
for sub_directory in ("model_lists", "build", "logs"):
    (COMPILATION_DIRECTORY / sub_directory).mkdir(parents=True, exist_ok=True)
compilation_environment = toolchain.environment(COMPILATION_DIRECTORY / "toolchain")
print(f"Compilation directory: {COMPILATION_DIRECTORY}")

In [ ]:
# --- 7. WRITE THE ARCHITECTURE FILES ---
# An architecture file (model list) describes the model to compile: here a preassembled
# model made of the machine, its voltage regulator and its governor, with the connections
# between them (connect) and between their initialization models (initConnect). The id of
# the modelicaModel is the name of the compiled library, and the id of each unit dynamic
# model is the prefix of its parameters and variables (generator_, voltageRegulator_...).
STANDARD_MODEL = "GeneratorSynchronousThreeWindingsGoverNordicVRNordic"
VARIANT_MODEL = "GeneratorSynchronousThreeWindingsGoverNordicVRProportional"

ARCHITECTURES = {
    # Definition of the Dynawo 1.7.0 model database
    STANDARD_MODEL: """<?xml version='1.0' encoding='UTF-8'?>
<dyn:dynamicModelsArchitecture xmlns:dyn="http://www.rte-france.com/dynawo">
  <dyn:modelicaModel id="GeneratorSynchronousThreeWindingsGoverNordicVRNordic">
    <dyn:unitDynamicModel id="generator" name="Dynawo.Electrical.Machines.OmegaRef.GeneratorSynchronous" initName="Dynawo.Electrical.Machines.OmegaRef.GeneratorSynchronousExt3W_INIT"/>
    <dyn:unitDynamicModel id="voltageRegulator" name="Dynawo.Electrical.Controls.Machines.VoltageRegulators.Simplified.VRNordic" initName="Dynawo.Electrical.Controls.Machines.VoltageRegulators.Simplified.VRNordic_INIT"/>
    <dyn:unitDynamicModel id="governor" name="Dynawo.Electrical.Controls.Machines.Governors.Simplified.GoverNordic" initName="Dynawo.Electrical.Controls.Machines.Governors.Governor_INIT"/>
    <dyn:initConnect id1="voltageRegulator" var1="Efd0Pu" id2="generator" var2="Efd0Pu"/>
    <dyn:initConnect id1="voltageRegulator" var1="Ir0Pu" id2="generator" var2="IRotor0Pu"/>
    <dyn:initConnect id1="voltageRegulator" var1="Us0Pu" id2="generator" var2="UStator0Pu"/>
    <dyn:initConnect id1="governor" var1="Pm0Pu" id2="generator" var2="Pm0Pu"/>
    <dyn:connect id1="voltageRegulator" var1="efdPu" id2="generator" var2="efdPu.value"/>
    <dyn:connect id1="voltageRegulator" var1="IrPu" id2="generator" var2="IRotorPu.value"/>
    <dyn:connect id1="voltageRegulator" var1="omegaPu" id2="generator" var2="omegaPu.value"/>
    <dyn:connect id1="voltageRegulator" var1="UsPu" id2="generator" var2="UStatorPu.value"/>
    <dyn:connect id1="governor" var1="omegaPu" id2="generator" var2="omegaPu.value"/>
    <dyn:connect id1="governor" var1="PGenPu" id2="generator" var2="PGenPu"/>
    <dyn:connect id1="governor" var1="PmPu" id2="generator" var2="PmPu.value"/>
  </dyn:modelicaModel>
</dyn:dynamicModelsArchitecture>
""",
    # Variant: the same machine and governor, with the proportional voltage regulator (its
    # connections are those of the Dynawo models with proportional regulations)
    VARIANT_MODEL: """<?xml version='1.0' encoding='UTF-8'?>
<dyn:dynamicModelsArchitecture xmlns:dyn="http://www.rte-france.com/dynawo">
  <dyn:modelicaModel id="GeneratorSynchronousThreeWindingsGoverNordicVRProportional">
    <dyn:unitDynamicModel id="generator" name="Dynawo.Electrical.Machines.OmegaRef.GeneratorSynchronous" initName="Dynawo.Electrical.Machines.OmegaRef.GeneratorSynchronousExt3W_INIT"/>
    <dyn:unitDynamicModel id="voltageRegulator" name="Dynawo.Electrical.Controls.Machines.VoltageRegulators.Simplified.VRProportional" initName="Dynawo.Electrical.Controls.Machines.VoltageRegulators.Simplified.VRProportional_INIT"/>
    <dyn:unitDynamicModel id="governor" name="Dynawo.Electrical.Controls.Machines.Governors.Simplified.GoverNordic" initName="Dynawo.Electrical.Controls.Machines.Governors.Governor_INIT"/>
    <dyn:initConnect id1="voltageRegulator" var1="Efd0PuLF" id2="generator" var2="Efd0Pu"/>
    <dyn:initConnect id1="voltageRegulator" var1="Us0Pu" id2="generator" var2="UStator0Pu"/>
    <dyn:initConnect id1="governor" var1="Pm0Pu" id2="generator" var2="Pm0Pu"/>
    <dyn:connect id1="voltageRegulator" var1="EfdPu" id2="generator" var2="efdPu.value"/>
    <dyn:connect id1="voltageRegulator" var1="UsPu" id2="generator" var2="UStatorPu.value"/>
    <dyn:connect id1="governor" var1="omegaPu" id2="generator" var2="omegaPu.value"/>
    <dyn:connect id1="governor" var1="PGenPu" id2="generator" var2="PGenPu"/>
    <dyn:connect id1="governor" var1="PmPu" id2="generator" var2="PmPu.value"/>
  </dyn:modelicaModel>
</dyn:dynamicModelsArchitecture>
""",
}

for model_name, architecture in ARCHITECTURES.items():
    (COMPILATION_DIRECTORY / "model_lists" / f"{model_name}.xml").write_text(architecture)
print(ARCHITECTURES[VARIANT_MODEL])

In [ ]:
# --- 8. COMPILE THE MODELS ---
def compile_model(model_name):
    """
    Compiles a model with the Dynawo model generator, then dumps its description, which
    also checks that Dynawo can load the library. A library compiled from the same
    architecture file is reused.
    """
    model_list = COMPILATION_DIRECTORY / "model_lists" / f"{model_name}.xml"
    library = COMPILATION_DIRECTORY / f"{model_name}.so"
    description = COMPILATION_DIRECTORY / f"{model_name}.desc.xml"
    compiled_list = COMPILATION_DIRECTORY / "build" / f"{model_name}.xml"
    if (
        description.exists()
        and compiled_list.exists()
        and compiled_list.read_text() == model_list.read_text()
    ):
        print(f"{model_name}: up to date, reused.")
        return

    launcher = str(toolchain.launcher)
    build_directory = COMPILATION_DIRECTORY / "build" / model_name
    shutil.rmtree(build_directory, ignore_errors=True)
    build_directory.mkdir(parents=True)
    start = time.time()

    # 1. Preassembly: Modelica model -> C++ code -> library and its external variables
    command = [launcher, "jobs", "--generate-preassembled", "--model-list", str(model_list)]
    command += ["--output-dir", str(build_directory)]
    print("$ " + " ".join(command))
    # The return code is checked after the output is saved to the log
    process = subprocess.run(
        command, check=False, capture_output=True, text=True, env=compilation_environment
    )
    log_file = COMPILATION_DIRECTORY / "logs" / f"{model_name}.log"
    log_file.write_text(process.stdout + process.stderr)
    if process.returncode != 0:
        raise RuntimeError(f"Compilation of {model_name} failed, see {log_file}")
    for suffix in (".so", ".extvar"):
        shutil.move(
            build_directory / f"{model_name}{suffix}",
            COMPILATION_DIRECTORY / f"{model_name}{suffix}",
        )

    # 2. Description of the model: its parameters and variables
    command = [launcher, "jobs", "--dump-model", "--model-file", str(library)]
    command += ["--output-file", str(description)]
    print("$ " + " ".join(command))
    subprocess.run(command, capture_output=True, check=True, env=compilation_environment)

    shutil.copyfile(model_list, compiled_list)
    print(f"{model_name}: compiled in {time.time() - start:.0f} s.")


for model_name in ARCHITECTURES:
    compile_model(model_name)

print("\nCompiled models:")
for model_name in ARCHITECTURES:
    for path in sorted(COMPILATION_DIRECTORY.glob(f"{model_name}.*")):
        print(f"- {path.name}")

In [ ]:
# --- 9. INSPECT THE COMPILED VARIANT ---
def read_model_parameters(model_name):
    """Reads the writable parameters of a compiled model from its description (desc.xml)."""
    root = ET.parse(COMPILATION_DIRECTORY / f"{model_name}.desc.xml").getroot()
    return pd.DataFrame(
        [
            {
                "name": parameter.get("name"),
                "type": parameter.get("valueType"),
                "default": parameter.get("defaultValue"),
            }
            for parameter in root.iter("{http://www.rte-france.com/dynawo}parameter")
            if parameter.get("readOnly") == "false"
        ]
    )


# Parameters of the new voltage regulator: those without default value must be given in
# the parameter file (Step 6)
variant_parameters = read_model_parameters(VARIANT_MODEL)
display(variant_parameters[variant_parameters["name"].str.startswith("voltageRegulator_")])

In [ ]:
# --- 10. MODEL DATABASE OF THE SIMULATIONS ---
# PyPowSyBl cannot add model directories to the Dynawo jobs file it writes, and Dynawo
# refuses two libraries with the same name. The compiled models are therefore given to
# Dynawo through a model database of their own (DYNAWO_DDB_DIR): symbolic links to the
# model database of the installation, where the compiled models replace the standard
# models of the same name.
MODEL_DATABASE = COMPILATION_DIRECTORY / "model_database"
shutil.rmtree(MODEL_DATABASE, ignore_errors=True)
MODEL_DATABASE.mkdir()
compiled_files = {
    path.name: path
    for model_name in ARCHITECTURES
    for path in COMPILATION_DIRECTORY.glob(f"{model_name}.*")
}
for entry in (Path(dynawo_sys_path) / "ddb").iterdir():
    if entry.name not in compiled_files:
        (MODEL_DATABASE / entry.name).symlink_to(entry)
for name, path in compiled_files.items():
    (MODEL_DATABASE / name).symlink_to(path)
print(f"Model database: {MODEL_DATABASE} ({len(list(MODEL_DATABASE.iterdir()))} entries)")

# The variant is a new model for PyPowSyBl: it is registered as an additional model of the
# synchronous generator category
ADDITIONAL_MODELS_FILE = COMPILATION_DIRECTORY / "additional_models.json"
additional_models = {
    "SYNCHRONOUS_GENERATOR": {
        "libs": [
            {
                "lib": VARIANT_MODEL,
                "doc": "Three-winding synchronous generator with governor GoverNordic and "
                "voltage regulator VRProportional, compiled by this notebook.",
            }
        ]
    }
}
ADDITIONAL_MODELS_FILE.write_text(json.dumps(additional_models, indent=4))


@contextlib.contextmanager
def dynawo_model_database(model_database):
    """Makes the Dynawo runs of the block read their models from a model database."""
    previous = os.environ.get("DYNAWO_DDB_DIR")
    if model_database is not None:
        os.environ["DYNAWO_DDB_DIR"] = str(model_database)
    try:
        yield
    finally:
        if previous is None:
            os.environ.pop("DYNAWO_DDB_DIR", None)
        else:
            os.environ["DYNAWO_DDB_DIR"] = previous

## Step 5: Dynamic Models Setup

Map the static generators to their dynamic models. The precompiled and the compiled standard model share the name `GeneratorSynchronousThreeWindingsGoverNordicVRNordic`: the model database given to Dynawo decides which library is loaded. The variant has its own name and its own parameter sets.

In [ ]:
# --- 11. CONFIGURE DYNAMIC MODELS MAPPING ---
GENERATORS = ["GEN_1"]
VARIANT_SUFFIX = "_VRPROPORTIONAL"


def map_dynamic_models(model_name, parameter_suffix="") -> dyn.ModelMapping:
    """Maps the static generators to a dynamic model."""
    dynamic_mapping = dyn.ModelMapping()
    generator_mapping_df = pd.DataFrame(
        [
            {
                "static_id": generator,
                "parameter_set_id": f"{generator}{parameter_suffix}",
                "model_name": model_name,
            }
            for generator in GENERATORS
        ]
    ).set_index("static_id", drop=False)
    dynamic_mapping.add_synchronous_generator(generator_mapping_df)
    return dynamic_mapping


standard_mapping = map_dynamic_models(STANDARD_MODEL)
variant_mapping = map_dynamic_models(VARIANT_MODEL, VARIANT_SUFFIX)
print(f"Dynamic mapping complete for {len(GENERATORS)} generator(s).")

## Step 6: Physical Parameters and Loadflow Initialization

Construct the `.PAR` files. The standard model keeps the parameters of the original notebook. The variant shares the machine and governor parameters, and replaces those of `VRNordic` by those of `VRProportional` (values of the Dynawo IEEE14 examples). IIDM references (`<reference>`) retrieve $P$, $Q$, $V$, and $\theta$ (in rad, `angle_pu`) from the Loadflow solution to guarantee seamless $t=0$ initialization.

In [ ]:
# --- 12. GENERATE PHYSICAL PARAMETERS (PAR FILES) ---
# Machine and governor parameters, shared by the standard model and the variant
MACHINE_AND_GOVERNOR_PARAMETERS = """        <par type="DOUBLE" name="generator_DPu" value="0"/>
        <par type="INT" name="generator_ExcitationPu" value="1"/>
        <par type="DOUBLE" name="generator_H" value="3"/>
        <par type="DOUBLE" name="generator_MdPuEfd" value="1"/>
        <par type="DOUBLE" name="generator_PNomAlt" value="4275"/>
        <par type="DOUBLE" name="generator_PNomTurb" value="4275"/>
        <par type="DOUBLE" name="generator_RTfPu" value="0"/>
        <par type="DOUBLE" name="generator_RaPu" value="0.002"/>
        <par type="DOUBLE" name="generator_SNom" value="4500"/>
        <par type="DOUBLE" name="generator_SnTfo" value="4500"/>
        <par type="DOUBLE" name="generator_Tpd0" value="5"/>
        <par type="DOUBLE" name="generator_Tppd0" value="0.05"/>
        <par type="DOUBLE" name="generator_Tppq0" value="0.1"/>
        <par type="DOUBLE" name="generator_UBaseHV" value="15"/>
        <par type="DOUBLE" name="generator_UBaseLV" value="15"/>
        <par type="DOUBLE" name="generator_UNom" value="15"/>
        <par type="DOUBLE" name="generator_UNomHV" value="15"/>
        <par type="DOUBLE" name="generator_UNomLV" value="15"/>
        <par type="DOUBLE" name="generator_XTfPu" value="0"/>
        <par type="DOUBLE" name="generator_XdPu" value="1.1"/>
        <par type="DOUBLE" name="generator_XlPu" value="0.15"/>
        <par type="DOUBLE" name="generator_XpdPu" value="0.25"/>
        <par type="DOUBLE" name="generator_XppdPu" value="0.2"/>
        <par type="DOUBLE" name="generator_XppqPu" value="0.2"/>
        <par type="DOUBLE" name="generator_XqPu" value="0.7"/>
        <par type="DOUBLE" name="generator_md" value="0.1"/>
        <par type="DOUBLE" name="generator_mq" value="0.1"/>
        <par type="DOUBLE" name="generator_nd" value="6.0257"/>
        <par type="DOUBLE" name="generator_nq" value="6.0257"/>
        <par type="DOUBLE" name="governor_KSigma" value="0.08"/>
        <par type="DOUBLE" name="governor_Ki" value="0.4"/>
        <par type="DOUBLE" name="governor_Kp" value="2"/>
        <par type="DOUBLE" name="governor_PNom" value="4275"/>
        <par type="BOOL" name="generator_UseApproximation" value="true"/>
"""

# Parameters of the VRNordic voltage regulator (standard model)
VRNORDIC_PARAMETERS = """        <par type="DOUBLE" name="voltageRegulator_EfdMaxPu" value="4"/>
        <par type="DOUBLE" name="voltageRegulator_IrLimPu" value="1.8991"/>
        <par type="DOUBLE" name="voltageRegulator_KPss" value="0"/>
        <par type="DOUBLE" name="voltageRegulator_KTgr" value="70"/>
        <par type="DOUBLE" name="voltageRegulator_OelMode" value="0"/>
        <par type="DOUBLE" name="voltageRegulator_tDerOmega" value="1"/>
        <par type="DOUBLE" name="voltageRegulator_tLagPss" value="0.01"/>
        <par type="DOUBLE" name="voltageRegulator_tLagTgr" value="20"/>
        <par type="DOUBLE" name="voltageRegulator_tLeadPss" value="1"/>
        <par type="DOUBLE" name="voltageRegulator_tLeadTgr" value="10"/>
        <par type="DOUBLE" name="voltageRegulator_tOelMin" value="-11"/>
"""

# Parameters of the VRProportional voltage regulator (variant), from the Dynawo IEEE14 examples
VRPROPORTIONAL_PARAMETERS = """        <par type="DOUBLE" name="voltageRegulator_Gain" value="20"/>
        <par type="DOUBLE" name="voltageRegulator_EfdMinPu" value="-5"/>
        <par type="DOUBLE" name="voltageRegulator_EfdMaxPu" value="5"/>
        <par type="DOUBLE" name="voltageRegulator_LagEfdMax" value="0"/>
        <par type="DOUBLE" name="voltageRegulator_LagEfdMin" value="0"/>
        <par type="DOUBLE" name="voltageRegulator_UsRefMinPu" value="0.8"/>
        <par type="DOUBLE" name="voltageRegulator_UsRefMaxPu" value="1.2"/>
"""

# Initial values from the Loadflow solution: P, Q, V and the voltage angle in rad
INITIALIZATION_REFERENCES = """        <reference type="DOUBLE" name="generator_P0Pu" origData="IIDM" origName="p_pu"/>
        <reference type="DOUBLE" name="generator_Q0Pu" origData="IIDM" origName="q_pu"/>
        <reference type="DOUBLE" name="generator_U0Pu" origData="IIDM" origName="v_pu"/>
        <reference type="DOUBLE" name="generator_UPhase0" origData="IIDM" origName="angle_pu"/>
"""


def parameter_set(set_id, voltage_regulator_parameters):
    """Renders the parameter set of a generator with a voltage regulator."""
    return (
        f'    <set id="{set_id}">\n'
        + MACHINE_AND_GOVERNOR_PARAMETERS
        + voltage_regulator_parameters
        + INITIALIZATION_REFERENCES
        + "    </set>\n"
    )


def generate_parameter_files():
    """Writes the XML parameter files for the dynamic models and static network."""
    print("Generating dynamic physical parameters and Loadflow initialization mapping...")

    xml_dynamic_model_parameters = (
        '<?xml version="1.0" encoding="UTF-8"?>\n'
        '<parametersSet xmlns="http://www.rte-france.com/dynawo">\n'
    )
    for generator in GENERATORS:
        xml_dynamic_model_parameters += parameter_set(generator, VRNORDIC_PARAMETERS)
        xml_dynamic_model_parameters += parameter_set(
            f"{generator}{VARIANT_SUFFIX}", VRPROPORTIONAL_PARAMETERS
        )
    xml_dynamic_model_parameters += "</parametersSet>\n"

    xml_static_network_parameters = """<?xml version="1.0" encoding="UTF-8"?>
<parametersSet xmlns="http://www.rte-france.com/dynawo">
    <set id="Network">
        <par type="DOUBLE" name="capacitor_no_reclosing_delay" value="300.0"/>
        <par type="DOUBLE" name="dangling_line_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="line_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="load_Tp" value="90.0"/>
        <par type="DOUBLE" name="load_Tq" value="90.0"/>
        <par type="DOUBLE" name="load_alpha" value="1.0"/>
        <par type="DOUBLE" name="load_alphaLong" value="0.0"/>
        <par type="DOUBLE" name="load_beta" value="2.0"/>
        <par type="DOUBLE" name="load_betaLong" value="0.0"/>
        <par type="BOOL" name="load_isControllable" value="false"/>
        <par type="BOOL" name="load_isRestorative" value="false"/>
        <par type="DOUBLE" name="load_zPMax" value="100.0"/>
        <par type="DOUBLE" name="load_zQMax" value="100.0"/>
        <par type="DOUBLE" name="reactance_no_reclosing_delay" value="0.0"/>
        <par type="DOUBLE" name="transformer_currentLimit_maxTimeOperation" value="90.0"/>
        <par type="DOUBLE" name="transformer_t1st_HT" value="30.0"/>
        <par type="DOUBLE" name="transformer_t1st_THT" value="30.0"/>
        <par type="DOUBLE" name="transformer_tNext_HT" value="10.0"/>
        <par type="DOUBLE" name="transformer_tNext_THT" value="10.0"/>
        <par type="DOUBLE" name="transformer_tolV" value="0.014999999700000001"/>
        <par type="BOOL" name="BUS_LOAD_hasShortCircuitCapabilities" value="true"/>
    </set>
</parametersSet>
"""

    export_data_dir = "getting_started_data"
    os.makedirs(export_data_dir, exist_ok=True)

    with open(os.path.join(export_data_dir, "Base_Case.par"), "w") as file:
        file.write(xml_dynamic_model_parameters)

    with open(os.path.join(export_data_dir, "Network.par"), "w") as file:
        file.write(xml_static_network_parameters)

    print("Physical and network parameters successfully saved.")


# Execute file generation
generate_parameter_files()

## Step 7: Execute the Dynamic Simulations

Introduce the temporal fault (short-circuit) of the original notebook, and simulate it three times:

1. with the standard model precompiled in the Dynawo model database (as in the original notebook);
2. with the standard model compiled by this notebook: both give the same curves;
3. with the variant compiled by this notebook: the voltage recovery after the fault changes with the voltage regulator.

In [ ]:
# --- 13. EXECUTE THE DYNAMIC SIMULATIONS AND PLOT ---
# Observed curves of the original notebook: {curve column: label}
CURVES = {"NETWORK_BUS_LOAD_U_value": "Load bus", "NETWORK_BUS_GEN_U_value": "Generator bus"}


def run_simulation(mapping, model_database=None, additional_models_file=None):
    """Runs the short-circuit scenario with a dynamic mapping and returns the curves."""
    # Map events (Fault at BUS_LOAD)
    event_mapping = dyn.EventMapping()
    event_mapping.add_node_fault(
        static_id="BUS_LOAD", start_time=2.0, fault_time=0.1, r_pu=0.0, x_pu=0.0001
    )

    # Map output curves
    output_mapping = dyn.OutputVariableMapping()
    output_mapping.add_standard_model_curves("BUS_GEN", "U_value")
    output_mapping.add_standard_model_curves("BUS_LOAD", "U_value")
    # The other simulation parameters come from the PyPowSyBl configuration (Step 0)
    provider_parameters = {}
    if additional_models_file is not None:
        provider_parameters["additionalModelsFile"] = str(additional_models_file)
    sim_parameters = dyn.Parameters(
        start_time=0.0, stop_time=10.0, provider_parameters=provider_parameters
    )

    # PyPowSyBl writes the final state of a simulation into the network: every simulation
    # starts from a copy of the Loadflow solution
    network = pp.network.load_from_binary_buffer(
        power_network.save_to_binary_buffer(format="XIIDM")
    )
    with dynawo_model_database(model_database):
        sim_results = dyn.Simulation().run(
            network,
            model_mapping=mapping,
            event_mapping=event_mapping,
            timeseries_mapping=output_mapping,
            parameters=sim_parameters,
        )
    if sim_results.status().name != "SUCCESS":
        raise RuntimeError(f"Simulation failed. Status: {sim_results.status_text()}")
    return sim_results.curves()


def plot_simulation_results(precompiled_curves, compiled_curves, variant_curves):
    """Plots the voltage transient responses of the three simulations."""
    figure, axes = plt.subplots(1, 2, figsize=(15, 5), sharey=True)
    for column, label in CURVES.items():
        axes[0].plot(
            precompiled_curves.index, precompiled_curves[column], label=f"{label} (precompiled)"
        )
        axes[0].plot(
            compiled_curves.index,
            compiled_curves[column],
            linestyle="--",
            label=f"{label} (compiled)",
        )
        axes[1].plot(compiled_curves.index, compiled_curves[column], label=f"{label} (VRNordic)")
        axes[1].plot(
            variant_curves.index,
            variant_curves[column],
            linestyle="--",
            label=f"{label} (VRProportional)",
        )

    axes[0].set_title("Standard model: precompiled and compiled")
    axes[1].set_title("Compiled models: VRNordic and VRProportional")
    axes[0].set_ylabel("Voltage (kV)")
    for axis in axes:
        axis.set_xlabel("Time (Seconds)")
        axis.grid(True)
        axis.legend(fontsize=8)
    figure.suptitle("Transient Response: 2-Bus System Voltages")
    figure.tight_layout()
    plt.show()


print("1. Standard model precompiled in the Dynawo model database...")
precompiled_curves = run_simulation(standard_mapping)
print("2. Standard model compiled by this notebook...")
compiled_curves = run_simulation(standard_mapping, MODEL_DATABASE)
print("3. Variant compiled by this notebook (VRProportional voltage regulator)...")
variant_curves = run_simulation(variant_mapping, MODEL_DATABASE, ADDITIONAL_MODELS_FILE)

difference = abs(precompiled_curves.values - compiled_curves.values).max()
print(
    f"\nMaximum difference between the precompiled and the compiled standard model: {difference:.2e} kV"
)
plot_simulation_results(precompiled_curves, compiled_curves, variant_curves)